In [1]:
import io
import os
import json
import pandas as pd
import numpy as np
from datetime import datetime as dt
import statsmodels.api as sm
import statsmodels.formula.api as smf
import plotly.express as px
from IPython.display import HTML
from IPython.display import Image
import warnings
from sparky_bc import Sparky
import plotly.express as px
warnings.filterwarnings('ignore')

# files lz conection
path_sparky_conf = '/Users/santlond/Documents/sparky_conf.json'

# Configurar conexión a LZ
with open(path_sparky_conf, 'rb') as JSON_lz_File:
    sp_config = json.loads(JSON_lz_File.read())
    
USER='santlond'
PASS=sp_config['ID']
DSN='IMPALA_PROD'
LOGDIR= 'logs'
# sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp")
sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp", spark_submit="spark3-submit")
 
# sparky = Sparky(username=USER, password=PASS, dsn=DSN)

helper = sparky.helper

/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-06-15 18:26:14 - [WARNING] - No se encontro la carpeta "/Users/santlond/Documents/ADQUIRENCIA_FERIA_EVA/logs" para guardar los logs


 ____  _____ __  __  ___ _____ _____ 
|  _ \| ____|  \/  |/ _ \_   _| ____|
| |_) |  _| | |\/| | | | || | |  _|  
|  _ <| |___| |  | | |_| || | | |___ 
|_| \_\_____|_|  |_|\___/ |_| |_____|
                                     
 ____  ____   _    ____  _  __
/ ___||  _ \ / \  |  _ \| |/ /
\___ \| |_) / _ \ | |_) | ' / 
 ___) |  __/ ___ \|  _ <| . \ 
|____/|_| /_/   \_\_| \_\_|\_\
                              



# Fecha primer registro tabla wompi_merchants

In [22]:
dict_ult_ing_wompi_vinc_primer_regi_merch = helper.obtener_ultima_ingestion('resultados_wompi.wompi_merchants')
dict_ult_ing_wompi_vinc_primer_regi_merch

2026-06-15 19:06:55 - [INFO] - Buscando fechas para resultados_wompi.wompi_merchants
2026-06-15 19:06:56 - [INFO] - Finalizo la busqueda, duracion: 00:00.7, resultado: {'year': 2026, 'month': 6, 'day': 12}


{'year': 2026, 'month': 6, 'day': 12}

In [30]:
sql_drop = f"""DROP TABLE IF EXISTS proceso.mdo_wompi_vinc_primer_registro_merchants PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = f"""
CREATE TABLE proceso.mdo_wompi_vinc_primer_registro_merchants STORED AS PARQUET AS
SELECT id_comercio as codigo_unico,
              min(creado) AS fecha_primer_regi_merch
       FROM resultados_wompi.wompi_merchants
       WHERE YEAR = {dict_ult_ing_wompi_vinc_primer_regi_merch['year']}
       AND MONTH = {dict_ult_ing_wompi_vinc_primer_regi_merch['month']}
       AND DAY = {dict_ult_ing_wompi_vinc_primer_regi_merch['day']}
       AND modelo = 'Agregador'
       AND UPPER(activo) = 'A'
       AND UPPER(desembolsos_permitidos) = 'SI'
       AND UPPER(plan_dispersion) <> 'PLAN FREMIUM NEQUI NEGOCIOS'
       AND UPPER(plan_dispersion) <> 'PLAN NEQUI NEGOCIOS EMPRENDEDORES'
       AND creado <= 20260531 -- MODIFICAR. FECHA FINAL DE PRIMER REGISTRO DE LOS MERCHANTS A INCLUIR
       GROUP BY 1
"""
helper.ejecutar_consulta(sql)

sql_compute = f"""COMPUTE INCREMENTAL STATS proceso.mdo_wompi_vinc_primer_registro_merchants;"""
helper.ejecutar_consulta(sql_compute)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 28/28      DROP ..._wompi_vinc_primer_registro_merchants   finalizado   07:14:29 PM     00:00.2 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 29/29    CREATE ..._wompi_vinc_primer_registro_merchants   finalizado   07:14:29 PM     00:00.9 
-------------------------------------------------------------------------------------------------
--------------------

# Fecha primer registro tabla wompi_businesses_procedures

In [ ]:
dict_ult_ing_wompi_busines = helper.obtener_ultima_ingestion('resultados_wompi.wompi_businesses_procedures')
dict_ult_ing_wompi_busines

In [24]:
sql_drop = f"""DROP TABLE IF EXISTS proceso.mdo_wompi_vinc_primer_registro;"""
helper.ejecutar_consulta(sql_drop)

sql = f"""
CREATE TABLE proceso.mdo_wompi_vinc_primer_registro STORED AS PARQUET AS
SELECT id_comercio AS codigo_unico,
       min(fecha_actualizacion_procedimiento) AS fecha_primer_regi
FROM resultados_wompi.wompi_businesses_procedures
WHERE YEAR = {dict_ult_ing_wompi_busines['year']}
  AND MONTH = {dict_ult_ing_wompi_busines['month']}
  AND DAY = {dict_ult_ing_wompi_busines['day']}
GROUP BY 1
"""
helper.ejecutar_consulta(sql)

sql_compute = f"""COMPUTE INCREMENTAL STATS proceso.mdo_wompi_vinc_primer_registro"""
helper.ejecutar_consulta(sql_compute)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 18/18      DROP   proceso.mdo_wompi_vinc_primer_registro   finalizado   07:09:47 PM     00:00.2 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 19/19    CREATE   proceso.mdo_wompi_vinc_primer_registro   finalizado   07:09:47 PM     00:00.5 
-------------------------------------------------------------------------------------------------
--------------------

# Fecha vinculación completa

In [25]:
sql_drop = f"""DROP TABLE IF EXISTS proceso.mdo_wompi_vinc_completa;"""
helper.ejecutar_consulta(sql_drop)

sql = f"""
CREATE TABLE proceso.mdo_wompi_vinc_completa STORED AS PARQUET AS
SELECT id_comercio AS codigo_unico,
       min(fecha_actualizacion_procedimiento) AS fecha_vinc_completa
FROM resultados_wompi.wompi_businesses_procedures
WHERE YEAR = {dict_ult_ing_wompi_busines['year']}
  AND MONTH = {dict_ult_ing_wompi_busines['month']}
  AND DAY = {dict_ult_ing_wompi_busines['day']}
  AND id_procedimiento IN (2,
                           7,
                           19,
                           21,
                           124,
                           125)
  AND estado_comercio IN ('Activo con desembolsos',
                          'En Vinculacion Completa',
                          'En Vinculacion Temprana')
  AND estado_procedimiento = 'Aprobado'
GROUP BY 1;
"""
helper.ejecutar_consulta(sql)

sql_compute = f"""COMPUTE INCREMENTAL STATS proceso.mdo_wompi_vinc_completa"""
helper.ejecutar_consulta(sql_compute)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 21/21      DROP          proceso.mdo_wompi_vinc_completa   finalizado   07:09:50 PM     00:00.2 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 22/22    CREATE          proceso.mdo_wompi_vinc_completa   finalizado   07:09:50 PM     00:00.6 
-------------------------------------------------------------------------------------------------
--------------------

# Tabla para analisis

In [31]:
sql = f"""
SELECT m.codigo_unico,
       m.fecha_primer_regi_merch,
       a.fecha_primer_regi,
       b.fecha_vinc_completa,
       cast(left(cast(m.fecha_primer_regi_merch AS string), 4) AS int) AS y_primer_regi_merch,
       to_timestamp(concat(left(cast(m.fecha_primer_regi_merch AS string), 6), '01'), 'yyyyMMdd') AS ym_primer_regi_merch,
       datediff(to_timestamp(cast(b.fecha_vinc_completa AS STRING), 'yyyyMMdd'), to_timestamp(cast(m.fecha_primer_regi_merch AS STRING), 'yyyyMMdd')) AS dias_transcurridos
FROM proceso.mdo_wompi_vinc_primer_registro_merchants AS m
LEFT JOIN proceso.mdo_wompi_vinc_primer_registro AS a ON m.codigo_unico = a.codigo_unico
LEFT JOIN proceso.mdo_wompi_vinc_completa AS b ON m.codigo_unico = b.codigo_unico
"""
df = helper.obtener_dataframe(sql)

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 31/31 DATAFRAME                                           descargando   07:14:36 PM             

2026-06-15 19:14:41 - [INFO] - 144,866 filas, 7 columnas, 00:02.2 consultando, 00:02.5 descargando, 00:00.1 convirtiendo


 31/31 DATAFRAME                                            finalizado   07:14:36 PM     00:05.0 
-------------------------------------------------------------------------------------------------


In [32]:
df

,codigo_unico,fecha_primer_regi_merch,fecha_primer_regi,fecha_vinc_completa,y_primer_regi_merch,ym_primer_regi_merch,dias_transcurridos
0,113096,20220517,20220517,20220518.0,2022,2022-05-01,1.0
1,355467,20251211,20251211,NaN,2025,2025-12-01,NaN
2,395328,20260310,20260310,NaN,2026,2026-03-01,NaN
3,56519,20201130,20201130,20201201.0,2020,2020-11-01,1.0
4,140510,20230304,20230304,20230306.0,2023,2023-03-01,2.0
...,...,...,...,...,...,...,...
144861,116856,20220630,20220630,20231010.0,2022,2022-06-01,467.0
144862,411629,20260403,20260403,NaN,2026,2026-04-01,NaN
144863,43598,20200908,20200908,20200910.0,2020,2020-09-01,2.0
144864,367752,20260120,20260120,NaN,2026,2026-01-01,NaN


In [44]:
# Cliente ejemplo
df[df.codigo_unico == 100021]

,codigo_unico,fecha_primer_regi_merch,fecha_primer_regi,fecha_vinc_completa,y_primer_regi_merch,ym_primer_regi_merch,dias_transcurridos
70530,100021,20220110,20220110,20221003.0,2022,2022-01-01,266.0


In [ ]:
# El 98% de los códigos únicos tienen la misma fecha de primer registro en ambas tablas.
df[df.fecha_primer_regi_merch != df.fecha_primer_regi]

,codigo_unico,fecha_primer_regi_merch,fecha_primer_regi,fecha_vinc_completa,y_primer_regi_merch,ym_primer_regi_merch,dias_transcurridos
14,346332,20251119,20250929,NaN,2025,2025-11-01,NaN
63,378377,20260211,20260209,NaN,2026,2026-02-01,NaN
234,117943,20220714,20220715,20220715.0,2022,2022-07-01,1.0
422,300421,20250812,20250725,NaN,2025,2025-08-01,NaN
430,205361,20240228,20240306,NaN,2024,2024-02-01,NaN
...,...,...,...,...,...,...,...
144734,290698,20250620,20250619,NaN,2025,2025-06-01,NaN
144774,11490,20200425,20200426,20200430.0,2020,2020-04-01,5.0
144781,102184,20220131,20220204,20220214.0,2022,2022-01-01,14.0
144833,132198,20221213,20230307,20230310.0,2022,2022-12-01,87.0


In [65]:
# Adicionar columna para identificar codigos únicos con fecha de vinculacion completa
df['vinc_completa'] = df.fecha_vinc_completa.notnull() * 1
df


,codigo_unico,fecha_primer_regi_merch,fecha_primer_regi,fecha_vinc_completa,y_primer_regi_merch,ym_primer_regi_merch,dias_transcurridos,vinc_completa
0,113096,20220517,20220517,20220518.0,2022,2022-05-01,1.0,1
1,355467,20251211,20251211,NaN,2025,2025-12-01,NaN,0
2,395328,20260310,20260310,NaN,2026,2026-03-01,NaN,0
3,56519,20201130,20201130,20201201.0,2020,2020-11-01,1.0,1
4,140510,20230304,20230304,20230306.0,2023,2023-03-01,2.0,1
...,...,...,...,...,...,...,...,...
144861,116856,20220630,20220630,20231010.0,2022,2022-06-01,467.0,1
144862,411629,20260403,20260403,NaN,2026,2026-04-01,NaN,0
144863,43598,20200908,20200908,20200910.0,2020,2020-09-01,2.0,1
144864,367752,20260120,20260120,NaN,2026,2026-01-01,NaN,0


# Porcentaje de clientes listos para recibir transaccioens [vinculación completa]

In [66]:
round(df[~df.fecha_vinc_completa.isnull()].shape[0]/df.shape[0], 4)

0.2639

In [67]:
df.groupby(['y_primer_regi_merch'])['vinc_completa'].describe().reset_index()

,y_primer_regi_merch,count,mean,std,min,25%,50%,75%,max
0,2018,4.0,1.000000,0.000000,1.0,1.0,1.0,1.0,1.0
1,2019,1094.0,0.971664,0.166008,0.0,1.0,1.0,1.0,1.0
2,2020,11622.0,0.991137,0.093727,0.0,1.0,1.0,1.0,1.0
3,2021,10838.0,0.999908,0.009606,0.0,1.0,1.0,1.0,1.0
4,2022,10257.0,0.999415,0.024180,0.0,1.0,1.0,1.0,1.0
5,2023,13233.0,0.343384,0.474856,0.0,0.0,0.0,1.0,1.0
6,2024,24472.0,0.000286,0.016911,0.0,0.0,0.0,0.0,1.0
7,2025,40793.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
8,2026,32553.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0


In [70]:
fig = px.line(df.groupby(['y_primer_regi_merch'])['vinc_completa'].describe().reset_index(),
               x="y_primer_regi_merch", y="count", title='Evolución proporción de códigos únicos con vinculación completa por año')
fig.show()

In [71]:
fig = px.line(df.groupby(['y_primer_regi_merch'])['vinc_completa'].describe().reset_index(),
               x="y_primer_regi_merch", y="mean", title='Evolución proporción de códigos únicos con vinculación completa por año')
fig.show()

In [69]:
df.groupby(['ym_primer_regi_merch'])['vinc_completa'].describe().reset_index()

,ym_primer_regi_merch,count,mean,std,min,25%,50%,75%,max
0,2018-07-01,1.0,1.0,NaN,1.0,1.0,1.0,1.0,1.0
1,2018-10-01,1.0,1.0,NaN,1.0,1.0,1.0,1.0,1.0
2,2018-11-01,1.0,1.0,NaN,1.0,1.0,1.0,1.0,1.0
3,2018-12-01,1.0,1.0,NaN,1.0,1.0,1.0,1.0,1.0
4,2019-02-01,1.0,1.0,NaN,1.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...
87,2026-01-01,4701.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
88,2026-02-01,5692.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
89,2026-03-01,7124.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
90,2026-04-01,7816.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [72]:
fig = px.line(df.groupby(['ym_primer_regi_merch'])['vinc_completa'].describe().reset_index(),
               x="ym_primer_regi_merch", y="count", title='Evolución proporción de códigos únicos con vinculación completa por año')
fig.show()

In [73]:
fig = px.line(df.groupby(['ym_primer_regi_merch'])['vinc_completa'].describe().reset_index(),
               x="ym_primer_regi_merch", y="mean", title='Evolución proporción de códigos únicos con vinculación completa por año')
fig.show()

# Días trancurridos por año

In [37]:
df.groupby(['y_primer_regi_merch'])['dias_transcurridos'].describe().reset_index()

,y_primer_regi_merch,count,mean,std,min,25%,50%,75%,max
0,2018,4.0,371.000000,263.928020,150.0,195.0,298.5,474.5,737.0
1,2019,1063.0,146.635936,332.902343,0.0,2.0,6.0,43.5,1949.0
2,2020,11519.0,97.688949,252.448526,0.0,3.0,8.0,41.0,2221.0
3,2021,10837.0,38.199871,142.278563,0.0,1.0,1.0,6.0,1450.0
4,2022,10251.0,22.527363,90.659493,0.0,1.0,1.0,5.0,1082.0
5,2023,4544.0,18.566461,63.249173,0.0,1.0,2.0,5.0,672.0
6,2024,7.0,56.571429,103.251888,4.0,7.0,23.0,33.0,289.0
7,2025,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2026,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
fig = px.violin(df, y="dias_transcurridos", x="y_primer_regi_merch")
fig.show()

# Días transcurridos por mes

In [56]:
df_describe_ym = df[~df.fecha_vinc_completa.isnull()].groupby(['ym_primer_regi_merch'])['dias_transcurridos'].describe().reset_index()
df_describe_ym


,ym_primer_regi_merch,count,mean,std,min,25%,50%,75%,max
0,2018-07-01,1.0,387.0,NaN,387.0,387.00,387.0,387.00,387.0
1,2018-10-01,1.0,737.0,NaN,737.0,737.00,737.0,737.00,737.0
2,2018-11-01,1.0,210.0,NaN,210.0,210.00,210.0,210.00,210.0
3,2018-12-01,1.0,150.0,NaN,150.0,150.00,150.0,150.00,150.0
4,2019-02-01,1.0,420.0,NaN,420.0,420.00,420.0,420.00,420.0
...,...,...,...,...,...,...,...,...,...
63,2024-03-01,1.0,30.0,NaN,30.0,30.00,30.0,30.00,30.0
64,2024-04-01,1.0,289.0,NaN,289.0,289.00,289.0,289.00,289.0
65,2024-05-01,1.0,5.0,NaN,5.0,5.00,5.0,5.00,5.0
66,2024-06-01,2.0,13.5,13.435029,4.0,8.75,13.5,18.25,23.0


In [57]:
fig = px.line(df_describe_ym, x="ym_primer_regi_merch", y="count", title='Días transcurridos promedio entre primer registro y vinculación completa por mes de primer registro')
fig.show()

In [58]:
fig = px.line(df_describe_ym, x="ym_primer_regi_merch", y="mean", title='Días transcurridos promedio entre primer registro y vinculación completa por mes de primer registro')
fig.show()